In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from datetime import date, timedelta
import math
import numpy as np
import polars as pl
import torch
from torch import nn
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VAL_START = date(2020, 9, 9)
REFERENCE_DATE = VAL_START - timedelta(days=1)

K = 12
HISTORY_DAYS = 56
HALF_LIFE = 3

HISTORY_CANDIDATES = 50
SASREC_CANDIDATES = 100
DECAY_CANDIDATES = 100

print("Device:", DEVICE)

Device: cuda


In [3]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

NUM_ITEMS = article_mapping.height + 1

print("Validation users:", validation_ground_truth.height)
print("Items:", NUM_ITEMS - 1)

Validation users: 72019
Items: 105542


In [4]:
val_users = validation_ground_truth.select("customer_idx")

history_candidates = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=HISTORY_DAYS))
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"], descending=[False, True])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx")
        .unique(maintain_order=True)
        .head(HISTORY_CANDIDATES)
        .alias("history")
    )
    .collect()
)

history_map = dict(zip(
    history_candidates["customer_idx"].to_list(),
    history_candidates["history"].to_list()
))

In [5]:
decay_top100 = (
    train
    .with_columns(
        (pl.lit(REFERENCE_DATE) - pl.col("t_dat"))
        .dt.total_days()
        .cast(pl.Float32)
        .alias("age_days")
    )
    .with_columns(
        (-(pl.col("age_days") * math.log(2) / HALF_LIFE))
        .exp()
        .alias("weight")
    )
    .group_by("article_idx")
    .agg(pl.col("weight").sum().alias("score"))
    .sort("score", descending=True)
    .head(DECAY_CANDIDATES)
    .collect()["article_idx"]
    .to_list()
)

decay_top100[:12]

[103794,
 67523,
 67544,
 53893,
 104046,
 103797,
 3092,
 94675,
 101368,
 101719,
 103187,
 71111]

In [6]:
class SASRec(nn.Module):
    def __init__(self, num_items, max_len, hidden_dim, num_heads, num_layers, dropout):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.item_embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, input_items):
        seq_len = input_items.size(1)
        positions = torch.arange(seq_len, device=input_items.device).unsqueeze(0)

        x = self.item_embedding(input_items) * math.sqrt(self.hidden_dim)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_items.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        x = self.norm(x)

        return x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

In [7]:
checkpoint = torch.load(
    CHECKPOINTS_PATH / "sasrec_best.pt",
    map_location=DEVICE,
    weights_only=False
)

model = SASRec(
    num_items=NUM_ITEMS,
    max_len=checkpoint["max_len"],
    hidden_dim=checkpoint["hidden_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

MAX_LEN = checkpoint["max_len"]

print("Best SASRec epoch:", checkpoint["epoch"])
print("Best SASRec MAP@12:", checkpoint["metrics"]["MAP@12"])

Best SASRec epoch: 8
Best SASRec MAP@12: 0.014934833159714998


In [8]:
sasrec_history = (
    train
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").tail(MAX_LEN).alias("sequence"))
    .collect()
)

sasrec_history_map = dict(zip(
    sasrec_history["customer_idx"].to_list(),
    sasrec_history["sequence"].to_list()
))

In [9]:
val_user_ids = validation_ground_truth["customer_idx"].to_numpy()

val_sequences = np.zeros((len(val_user_ids), MAX_LEN), dtype=np.int64)
has_history = np.zeros(len(val_user_ids), dtype=bool)

for i, user_id in enumerate(val_user_ids):
    sequence = sasrec_history_map.get(int(user_id), [])

    if sequence:
        sequence = sequence[-MAX_LEN:]
        val_sequences[i, -len(sequence):] = sequence
        has_history[i] = True

print("Validation users:", len(val_user_ids))
print("With history:", has_history.sum())
print("Without history:", (~has_history).sum())

Validation users: 72019
With history: 66624
Without history: 5395


In [10]:
candidate_items = (
    train
    .select("article_idx")
    .unique()
    .collect()["article_idx"]
    .to_numpy()
)

candidate_items_tensor = torch.from_numpy(
    candidate_items.astype(np.int64)
).to(DEVICE)

print("Candidate items:", len(candidate_items))

Candidate items: 102967


In [11]:
EVAL_BATCH_SIZE = 256
sasrec_map = {}

model.eval()

with torch.no_grad():
    item_embeddings = model.item_embedding(candidate_items_tensor)

    for start in tqdm(
        range(0, len(val_user_ids), EVAL_BATCH_SIZE),
        desc="SASRec candidates"
    ):
        end = min(start + EVAL_BATCH_SIZE, len(val_user_ids))

        batch_user_ids = val_user_ids[start:end]
        batch_sequences = val_sequences[start:end]
        batch_has_history = has_history[start:end]

        valid_indices = np.flatnonzero(batch_has_history)

        if len(valid_indices) == 0:
            continue

        sequence_tensor = torch.from_numpy(
            batch_sequences[valid_indices]
        ).to(DEVICE)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            hidden = model(sequence_tensor)
            user_embeddings = hidden[:, -1]
            scores = user_embeddings @ item_embeddings.T

        top_indices = scores.topk(
            SASREC_CANDIDATES,
            dim=1
        ).indices

        recommendations = candidate_items_tensor[
            top_indices
        ].cpu().tolist()

        for position, recommendation in zip(
            valid_indices,
            recommendations
        ):
            sasrec_map[int(batch_user_ids[position])] = recommendation

print("Users with SASRec candidates:", len(sasrec_map))

SASRec candidates:   0%|          | 0/282 [00:00<?, ?it/s]

Users with SASRec candidates: 66624


## Weighted Rank Fusion

$$
\mathrm{score}(i) =
\frac{w_h}{c + r_h(i)}
+
\frac{w_s}{c + r_s(i)}
+
\frac{w_d}{c + r_d(i)}
$$

In [12]:
RRF_K = 20

def fuse_recommendations(
    user_id,
    history_weight,
    sasrec_weight,
    decay_weight,
    k=12
):
    scores = {}

    sources = [
        (history_map.get(user_id, []), history_weight),
        (sasrec_map.get(user_id, []), sasrec_weight),
        (decay_top100, decay_weight)
    ]

    for items, weight in sources:
        for rank, item in enumerate(items, 1):
            scores[item] = scores.get(item, 0.0) + weight / (RRF_K + rank)

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return ranked[:k]

In [13]:
example_user = int(val_user_ids[0])

print("History:", history_map.get(example_user, [])[:10])
print("SASRec:", sasrec_map.get(example_user, [])[:10])
print("DecayPop:", decay_top100[:10])

print(
    "Hybrid:",
    fuse_recommendations(
        example_user,
        history_weight=1.0,
        sasrec_weight=0.5,
        decay_weight=0.25
    )
)

History: [97494, 13747]
SASRec: [103792, 94658, 101368, 104363, 103865, 104361, 100646, 103795, 100648, 104362]
DecayPop: [103794, 67523, 67544, 53893, 104046, 103797, 3092, 94675, 101368, 101719]
Hybrid: [97494, 13747, 101368, 94658, 103792, 103795, 103865, 103794, 101719, 104363, 104361, 100646]


In [14]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i

        seen.add(item)

    return score / min(len(actual), k)

In [15]:
actuals = validation_ground_truth["actual"].to_list()

In [16]:
def evaluate_weights(history_weight, sasrec_weight, decay_weight):
    predictions = [
        fuse_recommendations(
            int(user_id),
            history_weight,
            sasrec_weight,
            decay_weight,
            K
        )
        for user_id in val_user_ids
    ]

    map12 = sum(
        average_precision_at_k(actual, predicted, K)
        for actual, predicted in zip(actuals, predictions)
    ) / len(actuals)

    return map12

In [17]:
weight_results = []

for sasrec_weight in [0.25, 0.5, 0.75, 1.0]:
    for decay_weight in [0.1, 0.25, 0.5]:
        map12 = evaluate_weights(
            history_weight=1.0,
            sasrec_weight=sasrec_weight,
            decay_weight=decay_weight
        )

        weight_results.append({
            "history_weight": 1.0,
            "sasrec_weight": sasrec_weight,
            "decay_weight": decay_weight,
            "MAP@12": map12
        })

        print(
            f"history=1.0 "
            f"sasrec={sasrec_weight} "
            f"decay={decay_weight} "
            f"MAP@12={map12:.6f}"
        )

history=1.0 sasrec=0.25 decay=0.1 MAP@12=0.026338
history=1.0 sasrec=0.25 decay=0.25 MAP@12=0.026870
history=1.0 sasrec=0.25 decay=0.5 MAP@12=0.025968
history=1.0 sasrec=0.5 decay=0.1 MAP@12=0.026004
history=1.0 sasrec=0.5 decay=0.25 MAP@12=0.026211
history=1.0 sasrec=0.5 decay=0.5 MAP@12=0.026573
history=1.0 sasrec=0.75 decay=0.1 MAP@12=0.025821
history=1.0 sasrec=0.75 decay=0.25 MAP@12=0.025875
history=1.0 sasrec=0.75 decay=0.5 MAP@12=0.025488
history=1.0 sasrec=1.0 decay=0.1 MAP@12=0.024912
history=1.0 sasrec=1.0 decay=0.25 MAP@12=0.024754
history=1.0 sasrec=1.0 decay=0.5 MAP@12=0.024101


In [18]:
weight_results = pl.DataFrame(weight_results).sort(
    "MAP@12",
    descending=True
)

weight_results

history_weight,sasrec_weight,decay_weight,MAP@12
f64,f64,f64,f64
1.0,0.25,0.25,0.02687
1.0,0.5,0.5,0.026573
1.0,0.25,0.1,0.026338
1.0,0.5,0.25,0.026211
1.0,0.5,0.1,0.026004
…,…,…,…
1.0,0.75,0.1,0.025821
1.0,0.75,0.5,0.025488
1.0,1.0,0.1,0.024912


In [19]:
fine_results = []

for sasrec_weight in [0.15, 0.20, 0.25, 0.30, 0.35]:
    for decay_weight in [0.15, 0.20, 0.25, 0.30, 0.35]:
        map12 = evaluate_weights(
            history_weight=1.0,
            sasrec_weight=sasrec_weight,
            decay_weight=decay_weight
        )

        fine_results.append({
            "history_weight": 1.0,
            "sasrec_weight": sasrec_weight,
            "decay_weight": decay_weight,
            "MAP@12": map12
        })

fine_results = pl.DataFrame(fine_results).sort("MAP@12", descending=True)

fine_results

history_weight,sasrec_weight,decay_weight,MAP@12
f64,f64,f64,f64
1.0,0.15,0.15,0.027052
1.0,0.2,0.2,0.027006
1.0,0.25,0.25,0.02687
1.0,0.2,0.15,0.02682
1.0,0.3,0.3,0.026796
…,…,…,…
1.0,0.3,0.15,0.026355
1.0,0.35,0.15,0.026269
1.0,0.2,0.35,0.026218


In [20]:
best_weights = fine_results.row(0, named=True)

HISTORY_WEIGHT = best_weights["history_weight"]
SASREC_WEIGHT = best_weights["sasrec_weight"]
DECAY_WEIGHT = best_weights["decay_weight"]

print("History weight:", HISTORY_WEIGHT)
print("SASRec weight:", SASREC_WEIGHT)
print("Decay weight:", DECAY_WEIGHT)
print("MAP@12:", best_weights["MAP@12"])

History weight: 1.0
SASRec weight: 0.15
Decay weight: 0.15
MAP@12: 0.027051584421777745


In [21]:
hybrid_predictions = [
    fuse_recommendations(
        int(user_id),
        HISTORY_WEIGHT,
        SASREC_WEIGHT,
        DECAY_WEIGHT,
        K
    )
    for user_id in val_user_ids
]

In [22]:
def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(
        1 / math.log2(i + 2)
        for i, item in enumerate(predicted[:k])
        if item in actual
    )

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(min(len(actual), k))
    )

    return dcg / idcg

In [23]:
hybrid_metrics = {
    "MAP@12": sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, hybrid_predictions)
    ) / len(actuals),

    "Recall@12": sum(
        recall_at_k(a, p, K)
        for a, p in zip(actuals, hybrid_predictions)
    ) / len(actuals),

    "NDCG@12": sum(
        ndcg_at_k(a, p, K)
        for a, p in zip(actuals, hybrid_predictions)
    ) / len(actuals),

    "Coverage": len({
        item
        for prediction in hybrid_predictions
        for item in prediction
    }) / (NUM_ITEMS - 1)
}

hybrid_metrics

{'MAP@12': 0.027051584421777745,
 'Recall@12': 0.05531190264815282,
 'NDCG@12': 0.0395756649298745,
 'Coverage': 0.25318830418222127}

In [24]:
comparison = pl.DataFrame([
    {
        "model": "Personal history 56d + DecayPop",
        "MAP@12": 0.025264,
        "Recall@12": 0.050229,
        "NDCG@12": 0.036482,
        "Coverage": 0.218965
    },
    {
        "model": "SASRec",
        "MAP@12": 0.014935,
        "Recall@12": 0.036801,
        "NDCG@12": 0.024128,
        "Coverage": 0.168341
    },
    {
        "model": "Hybrid",
        **hybrid_metrics
    }
]).sort("MAP@12", descending=True)

comparison

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Hybrid""",0.027052,0.055312,0.039576,0.253188
"""Personal history 56d + DecayPo…",0.025264,0.050229,0.036482,0.218965
"""SASRec""",0.014935,0.036801,0.024128,0.168341
